# 🔹 Aletas e Trocadores de Calor
## Volume II – Capítulos 4-5 | Nível: Graduação/Pós-Graduação

**Objetivos de aprendizagem:**
- Resolver a equação da aleta e calcular eficiência/efetividade
- Aplicar métodos LMTD e ε-NTU para trocadores
- Dimensionar superfícies de troca térmica
- Analisar trade-offs entre ganho térmico e custo

**Referência:** Lugon Junior (2026), Vol II, Cap. 4-5.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 🔸 Eficiência de Aleta Retangular

In [ ]:
def eficiencia_aleta_retangular(h, P, k, Ac, L):
    """
    Calcula eficiência de aleta retangular (ponta adiabática).
    """
    m = np.sqrt(h * P / (k * Ac))
    Lc = L + 0.5 * np.sqrt(Ac)  # comprimento corrigido
    return np.tanh(m * Lc) / (m * Lc)

def efetividade_aleta(h, P, k, Ac, L, Ab):
    """
    Calcula efetividade da aleta.
    """
    m = np.sqrt(h * P / (k * Ac))
    q_fin = np.sqrt(h * P * k * Ac) * np.tanh(m * L)
    return q_fin / (h * Ab)

# Exercício Vol II, Cap 4, Ex 1
k = 205          # W/(m·K) - alumínio
L = 0.040        # 40 mm
t = 0.003        # 3 mm
w = 0.060        # 60 mm
h = 50           # W/(m²·K)

Ac = t * w
P = 2 * (t + w)
Ab = t * w  # área da base

eta = eficiencia_aleta_retangular(h, P, k, Ac, L)
eps = efetividade_aleta(h, P, k, Ac, L, Ab)

print(f"Aleta retangular de alumínio:")
print(f"  Dimensões: {L*1000:.0f}×{w*1000:.0f}×{t*1000:.0f} mm")
print(f"  Eficiência: η_fin = {eta*100:.1f}%")
print(f"  Efetividade: ε_fin = {eps:.2f}")
if eps > 2:
    print(f"  → Aleta JUSTIFICÁVEL ✓")
else:
    print(f"  → Aleta NÃO justificada (ε_fin < 2)")

## 🔸 Método LMTD para Trocadores

In [ ]:
def LMTD_contracorrente(Th_in, Th_out, Tc_in, Tc_out):
    """Diferença de temperatura média logarítmica (contracorrente)."""
    DT1 = Th_in - Tc_out
    DT2 = Th_out - Tc_in
    if DT1 == DT2:
        return DT1
    return (DT1 - DT2) / np.log(DT1 / DT2)

# Exercício Vol II, Cap 5, Ex 1
Th_in, Th_out = 150, 90   # °C
Tc_in, Tc_out = 30, 70    # °C

DT_lm = LMTD_contracorrente(Th_in, Th_out, Tc_in, Tc_out)
DT_media = ((Th_in - Tc_out) + (Th_out - Tc_in)) / 2

print(f"\nTrocador em contracorrente:")
print(f"  ΔT₁ = {Th_in - Tc_out:.0f} K, ΔT₂ = {Th_out - Tc_in:.0f} K")
print(f"  LMTD = {DT_lm:.1f} K")
print(f"  Média aritmética = {DT_media:.1f} K")
print(f"  Diferença: {(DT_media - DT_lm)/DT_lm*100:.1f}%")

## 🔸 Método ε-NTU

In [ ]:
def epsilon_NTU_contracorrente(NTU, Cr):
    """Efetividade para trocador em contracorrente."""
    if Cr == 1:
        return NTU / (1 + NTU)
    return (1 - np.exp(-NTU * (1 - Cr))) / (1 - Cr * np.exp(-NTU * (1 - Cr)))

# Exercício Vol II, Cap 5, Ex 2
U = 450          # W/(m²·K)
A = 12.0         # m²
C_min = 2000     # W/K
Cr = 0.5         # C_min/C_max
DT_in = 80       # K (Th,in - Tc,in)

NTU = U * A / C_min
eps = epsilon_NTU_contracorrente(NTU, Cr)
Q_max = C_min * DT_in
Q = eps * Q_max

print(f"\nTrocador casco-tubo (ε-NTU):")
print(f"  NTU = {NTU:.2f}")
print(f"  Efetividade: ε = {eps:.3f}")
print(f"  Q_max = {Q_max/1000:.1f} kW")
print(f"  Q_real = {Q/1000:.1f} kW")

# Varredura: ε vs NTU para diferentes Cr
NTU_vals = np.linspace(0, 5, 100)
Cr_vals = [0, 0.25, 0.5, 0.75, 1.0]

plt.figure(figsize=(8, 5))
for Cr in Cr_vals:
    eps_vals = [epsilon_NTU_contracorrente(NTU, Cr) for NTU in NTU_vals]
    plt.plot(NTU_vals, eps_vals, label=f'Cr = {Cr}')

plt.xlabel('NTU')
plt.ylabel('Efetividade ε')
plt.title('Curvas ε-NTU para trocador em contracorrente')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 🔸 Caso de Estudo: Dissipador para Processador

In [ ]:
def dimensionar_dissipador(P_chip, T_max, T_amb, h, k, N, L, t, w, s, tb, Ab):
    """
    Dimensiona dissipador de calor para processador.
    
    Retorna:
    --------
    dict : Resultados do dimensionamento
    """
    # Parâmetros de uma aleta
    Ac = t * w
    P = 2 * (t + w)
    Lc = L + t/2
    
    m = np.sqrt(h * P / (k * Ac))
    eta = np.tanh(m * Lc) / (m * Lc)
    
    # Calor por aleta
    A_fin = P * L
    theta_b_est = 60  # estimativa inicial [K]
    q_fin = eta * h * A_fin * theta_b_est
    
    # Calor da base exposta
    A_base_exp = Ab - N * (t * w)
    q_base = h * A_base_exp * theta_b_est
    
    # Calor total
    q_total = N * q_fin + q_base
    
    # Resistência térmica e temperatura do chip
    R_total = theta_b_est / q_total
    T_chip = T_amb + P_chip * R_total
    
    return {
        'eta_fin': eta,
        'q_total': q_total,
        'R_total': R_total,
        'T_chip': T_chip,
        'margem': T_max - T_chip
    }

# Parâmetros do caso de estudo (Vol II, Cap 4)
P_chip = 95      # W
T_max = 85 + 273.15  # 85°C → K
T_amb = 25 + 273.15  # 25°C → K
h = 75           # W/(m²·K)
k = 205          # W/(m·K) - alumínio
N = 12           # número de aletas
L, t, w = 0.030, 0.002, 0.050  # m
s = 0.003        # espaçamento
tb = 0.005       # espessura da base
Ab = 0.050 * 0.050  # área da base

res = dimensionar_dissipador(P_chip, T_max, T_amb, h, k, N, L, t, w, s, tb, Ab)

print(f"\nDissipador para processador:")
print(f"  Eficiência da aleta: η_fin = {res['eta_fin']*100:.1f}%")
print(f"  Calor total dissipado: q = {res['q_total']:.1f} W")
print(f"  Resistência térmica: R_total = {res['R_total']:.3f} K/W")
print(f"  Temperatura estimada do chip: T_chip = {res['T_chip']-273.15:.1f}°C")
print(f"  Margem de segurança: ΔT = {res['margem']:.1f} K")

if res['margem'] > 0:
    print(f"  → Dissipador ADEQUADO ✓")
else:
    print(f"  → Dissipador INSUFICIENTE (aumentar N ou L)")

## ✅ Exercícios Propostos

1. **[Graduação]** Calcule a efetividade para uma aleta cilíndrica de cobre (k=400 W/m·K, D=5 mm, L=25 mm) com h=80 W/m²·K.
2. **[Pós-Graduação]** Derive a equação da aleta para perfil triangular e mostre que a solução envolve funções de Bessel modificadas.

> 💡 Dica: Para aletas de perfil variável, use `scipy.special.iv` para funções de Bessel de primeira espécie.